# 04 — Where does a cold model load actually spend its time?

Tasks 15 and 16 observed whole-pipeline cold loads of ~9.4s and ~9.9s. Those
numbers say *that* a load is slow, not *why*. This notebook decomposes them,
to settle one question and calibrate two defaults.

**The question.** Is a cold load dominated by the PCIe copy — physics,
identical for any runtime including Ollama, nothing for embedx to build — or
by host-side work, which would be a real optimization opportunity? Either
answer is useful. "Nothing to optimize here" is a result.

**The defaults.** `default_keep_alive_s` (task 15) and
`max_concurrent_loads` (task 16) are currently justified guesses. The
reasoning is written down but the numbers are not measured. This notebook
proposes replacements. It does **not** change them: that is a follow-up,
once the numbers have been read.

## Four stages, never summed

| stage | what | measured |
|---|---|---|
| 1 | disk → OS page cache | **cold**, by evicting the files first |
| 2 | page cache → host RAM (`from_pretrained`) | **warm** page cache, deliberately |
| 3 | host RAM → device VRAM (the PCIe copy) | **warm** RAM, deliberately |
| 4 | first inference vs second (kernel autotune) | **cold** process, once each |

They are reported separately and never added into one "load" number. Adding
them would hide exactly the thing being measured.

## Warmup discipline — deliberately the opposite of notebook 03

Notebook 03 measured steady-state throughput, so warmups were run and
discarded. Here the cold/warm distinction **is** the measurement, so warmups
are never discarded; each stage states which side of the line it sits on and
why. Copying 03's pattern would delete the result.

**Every measurement runs in a fresh subprocess.** cuBLAS and cuDNN autotune
caches are per-process, so measuring stage 4 for several models inside one
kernel would have the first model pay the warmup and every later one look
free. A fresh process per measurement also guarantees a cold CUDA context
and no allocator reuse between models.

## Page-cache eviction, and why not `drop_caches`

Stage 1 needs a genuinely cold read. `sync; echo 3 > /proc/sys/vm/drop_caches`
needs root, and sudo on this host requires a password, so it was not used.

Instead each checkpoint's own files are evicted with
`posix_fadvise(POSIX_FADV_DONTNEED)`, which needs only read access. This is
better suited to the job than a global drop: it evicts the files under test
and leaves the rest of the system's cache alone, so the measurement does not
perturb everything else on the box.

It is verified rather than assumed — the eviction is confirmed against
`/proc/meminfo` `Cached:` on every run, and a run whose eviction did not take
is recorded as such rather than reported as cold. (A first attempt at this
measured a file under `/tmp`, which is **tmpfs**: fadvise can never evict
there because those pages *are* the memory. On the NVMe filesystem where the
HF cache lives it works, and the difference showed up as 13,376 MiB/s versus
1,158 MiB/s.)

In [1]:
# --- Environment record. Contaminated numbers that look clean are worse ---
# --- than no numbers, so this is checked, not assumed. ------------------
import json
import os
import pathlib
import shutil
import statistics
import subprocess
import sys
import tempfile
import time

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch

DRIVER = subprocess.run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip().splitlines()[0]
DEVICES = [
    {
        "index": i,
        "name": torch.cuda.get_device_name(i),
        "capability": list(torch.cuda.get_device_capability(i)),
        "total_memory_gib": torch.cuda.get_device_properties(i).total_memory / 2**30,
    }
    for i in range(torch.cuda.device_count())
]
for d in DEVICES:
    free, total = torch.cuda.mem_get_info(d["index"])
    d["free_gib_at_start"] = free / 2**30
    print(f"cuda:{d['index']} {d['name']}  cc{d['capability']}  "
          f"free {free / 2**30:.2f} / {total / 2**30:.2f} GiB")
print(f"driver {DRIVER}  torch {torch.__version__} (cuda {torch.version.cuda})")

# Who else holds VRAM? Read from nvidia-smi, not declared by hand.
procs = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory",
     "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
OTHER_VRAM_USERS = procs if procs else "none: no compute processes hold VRAM"
print(f"\nother VRAM users (from nvidia-smi): {OTHER_VRAM_USERS}")
ollama = subprocess.run(["pgrep", "-a", "ollama"], capture_output=True, text=True).stdout.strip()
print(f"ollama processes: {ollama or 'none'}")
print("(ollama runs on this host permanently; what matters is whether it holds "
      "VRAM, which the line above answers)")

Sat Aug  1 03:38:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 2000 Blac...    Off |   00000000:01:00.0  On |                  Off |
| 30%   34C    P8              7W /   70W |      34MiB /  16311MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

cuda:0 NVIDIA RTX PRO 2000 Blackwell  cc[12, 0]  free 15.32 / 15.48 GiB
cuda:1 NVIDIA RTX A400  cc[8, 6]  free 3.63 / 3.68 GiB
driver 610.43.02  torch 2.13.0+cu130 (cuda 13.0)

other VRAM users (from nvidia-smi): 152652, /home/luxor/Programming/embedx/.venv/bin/python3, 122 MiB
152652, /home/luxor/Programming/embedx/.venv/bin/python3, 44 MiB
ollama processes: 2078 /usr/local/bin/ollama serve
(ollama runs on this host permanently; what matters is whether it holds VRAM, which the line above answers)


## The worker

One subprocess per measurement, for the reasons in the header. It is written
out from this string rather than living in `src/embedx`, per task 17: nothing
here is needed at runtime, so it stays notebook-local. If a future `/info`
field wanted "how long did the last load take", *that* would justify moving a
helper into the package, with tests and the full gate.

Stages 2 and 3 use plain `AutoModel` rather than `SentenceTransformer`, so
that `from_pretrained` and `.to(device)` can be timed separately — the ST
wrapper does both in one call. embedx serves ST checkpoints through the ST
path, so the totals here are comparable to a real load but not identical to
it.

In [2]:
WORKER = r"""
import json, os, pathlib, sys, threading, time

def evict(paths):
    # Drop these files from the page cache. Returns MiB actually freed.
    def cached_kb():
        for line in open("/proc/meminfo"):
            if line.startswith("Cached:"):
                return int(line.split()[1])
        return -1
    before = cached_kb()
    for path in paths:
        fd = os.open(path, os.O_RDONLY)
        try:
            os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
        finally:
            os.close(fd)
    time.sleep(0.3)
    return (before - cached_kb()) / 1024.0

def read_all(paths):
    total = 0
    t0 = time.perf_counter()
    for path in paths:
        with open(path, "rb") as fh:
            while True:
                chunk = fh.read(16 << 20)
                if not chunk:
                    break
                total += len(chunk)
    return time.perf_counter() - t0, total

def link_state(index):
    import subprocess
    out = subprocess.run(
        ["nvidia-smi", "-i", str(index), "--format=csv,noheader",
         "--query-gpu=pcie.link.gen.current,pcie.link.width.current"],
        capture_output=True, text=True).stdout.strip()
    return "gen{}x{}".format(*[f.strip() for f in out.split(",")])

args = json.loads(sys.argv[1])
model_id, device_index = args["model_id"], args["device_index"]
snapshot = pathlib.Path(args["snapshot"])
weights = sorted(str(p) for p in snapshot.glob("*.safetensors"))
weight_bytes = sum(os.path.getsize(p) for p in weights)
result = {"model_id": model_id, "device_index": device_index,
          "weight_bytes": weight_bytes, "weight_files": len(weights)}

# --- Stage 1: disk -> OS page cache. COLD by construction. -------------
freed_mib = evict(weights)
result["evicted_mib"] = freed_mib
# An eviction that did not take would make this a warm read wearing a cold
# label, which is worse than no number at all.
result["eviction_verified"] = freed_mib > (weight_bytes / 2**20) * 0.5
s1, read_bytes = read_all(weights)
result["stage1_disk_to_page_cache_s"] = s1
result["stage1_gib_s"] = (read_bytes / 2**30) / s1

import torch
from transformers import AutoModel, AutoTokenizer

dtype = getattr(torch, args["dtype"])

# --- Stage 2: page cache -> host RAM. WARM page cache, deliberately: ----
# stage 1 just read these files, so this isolates from_pretrained's own
# work (safetensors parsing, tensor construction) from disk.
t0 = time.perf_counter()
model = AutoModel.from_pretrained(snapshot, dtype=dtype)
result["stage2_from_pretrained_cpu_s"] = time.perf_counter() - t0
# safetensors loads through mmap, so from_pretrained can return before the
# pages are actually faulted in. Touching every parameter prices that
# deferred work explicitly instead of letting it hide inside stage 3.
result["mmap_active"] = not bool(getattr(model.config, "disable_mmap", False))
t0 = time.perf_counter()
for p in model.parameters():
    p.data.sum().item()
result["stage2b_materialize_cpu_s"] = time.perf_counter() - t0
result["param_bytes"] = sum(p.numel() * p.element_size() for p in model.parameters())

# --- Stage 3: host RAM -> device VRAM. The PCIe copy. WARM RAM. --------
# Sampled, not assumed: both links park at Gen1 when idle and train up
# under load, which notebook 03 had to learn the hard way.
samples, sampling = [], True
def sample():
    while sampling:
        samples.append(link_state(device_index))
        time.sleep(0.02)
sampler = threading.Thread(target=sample, daemon=True)
sampler.start()
try:
    torch.cuda.synchronize(device_index)
    t0 = time.perf_counter()
    model = model.to(f"cuda:{device_index}")
    torch.cuda.synchronize(device_index)
    result["stage3_host_to_device_s"] = time.perf_counter() - t0
    result["oom"] = False
except torch.cuda.OutOfMemoryError as exc:
    result["stage3_host_to_device_s"] = None
    result["oom"] = True
    result["oom_detail"] = f"{type(exc).__name__}: {str(exc)[:200]}"
finally:
    sampling = False
    sampler.join(2)
result["pcie_link_during_stage3"] = sorted(set(samples))
if result["stage3_host_to_device_s"]:
    result["stage3_gib_s"] = (result["param_bytes"] / 2**30) / result["stage3_host_to_device_s"]

# --- Stage 4: first inference vs second. COLD process, once. -----------
# cuBLAS/cuDNN autotune is a one-time per-process cost, so this is a single
# observation per process by nature; repeating it in-process would only
# measure the warm case, which is what calls 2..N are for.
result["stage4_error"] = None
try:
  if not result["oom"]:
    tok = AutoTokenizer.from_pretrained(snapshot)
    batch = tok(["a short benchmark sentence"] * 8, return_tensors="pt",
                padding=True, truncation=True, max_length=128).to(f"cuda:{device_index}")
    model.eval()
    calls = []
    with torch.inference_mode():
        for _ in range(5):
            torch.cuda.synchronize(device_index)
            t0 = time.perf_counter()
            model(**batch)
            torch.cuda.synchronize(device_index)
            calls.append(time.perf_counter() - t0)
    result["stage4_first_embed_s"] = calls[0]
    result["stage4_warm_embed_s"] = min(calls[1:])
    result["stage4_all_calls_s"] = calls
    result["stage4_warmup_overhead_s"] = calls[0] - min(calls[1:])
except Exception as exc:
    # Stages 1-3 already succeeded; losing them because inference failed
    # would discard good measurements over an unrelated fault.
    result["stage4_error"] = f"{type(exc).__name__}: {str(exc)[:300]}"

result["ready_at"] = time.time()
print("RESULT " + json.dumps(result))
"""

# torch routes Qwen3's rotary embedding through a Triton kernel, and
# Triton JIT-compiles a C helper that needs Python.h. This host runs the
# distro python3.14 WITHOUT python3.14-dev, so that compile fails and any
# Qwen3 inference dies -- which is a real finding about this box, not just
# about this notebook: embedx serving Qwen3-Embedding-4B here would fail on
# its first request. Recorded in the results file.
#
# Rather than installing a system package (needs root), the headers come
# from a uv-managed CPython of the same minor version; 3.14.x is ABI-stable,
# so a module compiled against 3.14.6 headers loads under 3.14.4. This is a
# workaround for the benchmark, not a fix for the host.
UV_HEADERS = sorted(pathlib.Path.home().glob(
    ".local/share/uv/python/cpython-3.14*/include/python3.14"))
SYSTEM_HEADERS_PRESENT = pathlib.Path(
    f"/usr/include/python{sys.version_info.major}.{sys.version_info.minor}/Python.h").exists()
TRITON_ENV = {}
if not SYSTEM_HEADERS_PRESENT and UV_HEADERS:
    TRITON_ENV["CPATH"] = str(UV_HEADERS[-1])
TRITON_HEADER_NOTE = {
    "system_python_dev_headers_present": SYSTEM_HEADERS_PRESENT,
    "workaround_cpath": TRITON_ENV.get("CPATH"),
    "why": "torch dispatches Qwen3 RoPE to a Triton kernel; Triton JIT needs "
           "Python.h. Without python3.14-dev, Qwen3 inference fails on this "
           "host - for embedx in production too, not only for this notebook.",
}
print("python dev headers:", "system" if SYSTEM_HEADERS_PRESENT
      else f"MISSING - using {TRITON_ENV.get('CPATH')}")

WORKER_PATH = pathlib.Path(tempfile.gettempdir()) / "embedx_load_worker.py"
WORKER_PATH.write_text(WORKER)
print(f"worker written to {WORKER_PATH} ({len(WORKER)} bytes)")


def run_worker(model_id, snapshot, device_index, dtype="bfloat16", env=None):
    """One measurement, in its own process. Returns the parsed result."""
    args = json.dumps({"model_id": model_id, "snapshot": str(snapshot),
                       "device_index": device_index, "dtype": dtype})
    proc = subprocess.run([sys.executable, str(WORKER_PATH), args],
                          capture_output=True, text=True,
                          env={**os.environ, **TRITON_ENV, **(env or {})})
    for line in proc.stdout.splitlines():
        if line.startswith("RESULT "):
            return json.loads(line[len("RESULT "):])
    raise RuntimeError(f"worker produced no result:\n{proc.stdout[-2000:]}\n{proc.stderr[-2000:]}")

python dev headers: MISSING - using /home/luxor/.local/share/uv/python/cpython-3.14.6-linux-x86_64-gnu/include/python3.14
worker written to /tmp/embedx_load_worker.py (5521 bytes)


In [3]:
# --- Models under test. Nothing is downloaded here: a benchmark that ---
# --- fetches 8 GB as a side effect is not reproducible. ----------------
from huggingface_hub import snapshot_download

MODELS = [
    ("Qwen/Qwen3-Embedding-4B", "large"),
    ("intfloat/e5-small-v2", "small"),
    ("sentence-transformers/all-MiniLM-L6-v2", "small"),
]

SNAPSHOTS = {}
for model_id, _size in MODELS:
    path = pathlib.Path(snapshot_download(model_id, local_files_only=True))
    weights = sorted(path.glob("*.safetensors"))
    total = sum(p.stat().st_size for p in weights)
    SNAPSHOTS[model_id] = path
    print(f"{model_id:<45} {total / 2**30:6.2f} GiB in {len(weights)} safetensors file(s)")

FS = subprocess.run(["findmnt", "-no", "FSTYPE,SOURCE", str(next(iter(SNAPSHOTS.values())))],
                    capture_output=True, text=True).stdout.strip()
print(f"\nHF cache filesystem: {FS}")
print("(must not be tmpfs, or posix_fadvise cannot evict and stage 1 is a lie)")

Qwen/Qwen3-Embedding-4B                         7.49 GiB in 2 safetensors file(s)
intfloat/e5-small-v2                            0.12 GiB in 1 safetensors file(s)
sentence-transformers/all-MiniLM-L6-v2          0.08 GiB in 1 safetensors file(s)

HF cache filesystem: 
(must not be tmpfs, or posix_fadvise cannot evict and stage 1 is a lie)


/home/luxor/Programming/embedx/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Stages 1–4, per model and per device

Three repetitions for the large model and five for the small ones — enough
for a median without spending an hour on it. Every repetition is a fresh
process with a freshly evicted page cache, so every stage 1 is genuinely
cold and every stage 4 is genuinely a first inference.

In [4]:
RUNS = {"large": 3, "small": 5}
measurements = []

for model_id, size_class in MODELS:
    for device in DEVICES:
        runs = RUNS[size_class]
        print(f"\n{model_id} on cuda:{device['index']} ({runs} runs)")
        for n in range(runs):
            r = run_worker(model_id, SNAPSHOTS[model_id], device["index"])
            r["run"] = n
            r["size_class"] = size_class
            measurements.append(r)
            if r["oom"]:
                print(f"  run {n}: OOM on stage 3 — {r['oom_detail'][:80]}")
                continue
            print(f"  run {n}: s1 {r['stage1_disk_to_page_cache_s']:6.2f}s  "
                  f"s2 {r['stage2_from_pretrained_cpu_s']:6.2f}s  "
                  f"s2b {r['stage2b_materialize_cpu_s']:6.2f}s  "
                  f"s3 {r['stage3_host_to_device_s']:6.2f}s  "
                  f"s4 warmup {r['stage4_warmup_overhead_s'] * 1000:7.1f}ms  "
                  f"(evicted {r['evicted_mib']:.0f} MiB, verified={r['eviction_verified']})")
print(f"\n{len(measurements)} measurements collected")


Qwen/Qwen3-Embedding-4B on cuda:0 (3 runs)


  run 0: s1   6.38s  s2   0.23s  s2b   0.98s  s3   1.37s  s4 warmup  1201.4ms  (evicted 7671 MiB, verified=True)


  run 1: s1   6.58s  s2   0.20s  s2b   0.27s  s3   1.31s  s4 warmup  1181.1ms  (evicted 7671 MiB, verified=True)


  run 2: s1   6.38s  s2   0.20s  s2b   0.27s  s3   1.32s  s4 warmup  1192.8ms  (evicted 7671 MiB, verified=True)

Qwen/Qwen3-Embedding-4B on cuda:1 (3 runs)


  run 0: OOM on stage 3 — OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 1 has a t


...

  run 0: s1   0.07s  s2   0.14s  s2b   0.01s  s3   0.02s  s4 warmup   266.2ms  (evicted 87 MiB, verified=True)


  run 1: s1   0.08s  s2   0.15s  s2b   0.04s  s3   0.02s  s4 warmup   277.8ms  (evicted 87 MiB, verified=True)


  run 2: s1   0.10s  s2   0.14s  s2b   0.05s  s3   0.02s  s4 warmup   263.9ms  (evicted 86 MiB, verified=True)


  run 3: s1   0.09s  s2   0.14s  s2b   0.03s  s3   0.02s  s4 warmup   264.3ms  (evicted 87 MiB, verified=True)


  run 4: s1   0.10s  s2   0.14s  s2b   0.06s  s3   0.02s  s4 warmup   263.5ms  (evicted 87 MiB, verified=True)

26 measurements collected


In [5]:
# --- Medians per (model, device). Never summed into one number. -------
def median_of(rows, key):
    values = [r[key] for r in rows if r.get(key) is not None]
    return statistics.median(values) if values else None


STAGES = [
    ("stage1_disk_to_page_cache_s", "1 disk->cache"),
    ("stage2_from_pretrained_cpu_s", "2 from_pretrained"),
    ("stage2b_materialize_cpu_s", "2b materialize"),
    ("stage3_host_to_device_s", "3 host->VRAM"),
    ("stage4_warmup_overhead_s", "4 kernel warmup"),
]

summary = {}
for model_id, size_class in MODELS:
    for device in DEVICES:
        rows = [r for r in measurements
                if r["model_id"] == model_id and r["device_index"] == device["index"]]
        if not rows:
            continue
        key = f"{model_id}@cuda:{device['index']}"
        entry = {
            "model_id": model_id,
            "device_index": device["index"],
            "device_name": device["name"],
            "size_class": size_class,
            "runs": len(rows),
            "weight_bytes": rows[0]["weight_bytes"],
            "param_bytes": rows[0].get("param_bytes"),
            "oom": all(r["oom"] for r in rows),
            "eviction_verified_every_run": all(r["eviction_verified"] for r in rows),
            "mmap_active": rows[0].get("mmap_active"),
            "pcie_link_during_stage3": sorted({s for r in rows
                                               for s in r["pcie_link_during_stage3"]}),
        }
        entry["stage4_errors"] = sorted({r["stage4_error"] for r in rows
                                         if r.get("stage4_error")})
        for field, _label in STAGES:
            entry[f"median_{field}"] = median_of(rows, field)
        entry["median_stage3_gib_s"] = median_of(rows, "stage3_gib_s")
        entry["median_stage1_gib_s"] = median_of(rows, "stage1_gib_s")
        entry["median_stage4_first_embed_s"] = median_of(rows, "stage4_first_embed_s")
        entry["median_stage4_warm_embed_s"] = median_of(rows, "stage4_warm_embed_s")
        summary[key] = entry

header = f"{'model @ device':<52}" + "".join(f"{label:>18}" for _f, label in STAGES)
print(header)
print("-" * len(header))
for key, e in summary.items():
    if e["oom"]:
        print(f"{key:<52}" + f"{'OOM on stage 3':>18}")
        continue
    row = f"{key:<52}"
    for field, _label in STAGES:
        value = e[f"median_{field}"]
        row += f"{value:>17.3f}s" if value is not None else f"{'-':>18}"
    print(row)

model @ device                                           1 disk->cache 2 from_pretrained    2b materialize      3 host->VRAM   4 kernel warmup
----------------------------------------------------------------------------------------------------------------------------------------------
Qwen/Qwen3-Embedding-4B@cuda:0                                  6.382s            0.205s            0.268s            1.316s            1.193s
Qwen/Qwen3-Embedding-4B@cuda:1                          OOM on stage 3
intfloat/e5-small-v2@cuda:0                                     0.127s            0.169s            0.005s            0.020s            0.251s
intfloat/e5-small-v2@cuda:1                                     0.126s            0.169s            0.004s            0.035s            0.266s
sentence-transformers/all-MiniLM-L6-v2@cuda:0                   0.097s            0.143s            0.044s            0.013s            0.251s
sentence-transformers/all-MiniLM-L6-v2@cuda:1                   0.093s 

## Cross-check: stage 3 against notebook 03's measured H2D

Notebook 03 measured host-to-device bandwidth with pinned 256 MiB buffers:
6.36 GiB/s on cuda:0 (PCIe gen3 x8) and 2.81 GiB/s on cuda:1 (gen3 x4). The
host caps both cards at gen3 (`pcie.link.gen.hostmax = 3`), so those are the
ceilings here regardless of what the cards support.

If stage 3 is genuinely PCIe-bound, `param_bytes / measured_gib_s` should
land near the observed stage 3 time. A large gap is a finding, not something
to smooth over: it would mean `.to(device)` is doing something other than
streaming bytes across the bus.

In [6]:
NB03 = json.loads((pathlib.Path.cwd() / "dev" / "output" / "results.json").read_text()
                  if (pathlib.Path.cwd() / "dev").exists()
                  else (pathlib.Path.cwd().parent / "dev" / "output" / "results.json").read_text())
H2D = {int(k): v for k, v in NB03["h2d_gib_s"].items()}
print("notebook 03 H2D:", {k: round(v, 2) for k, v in H2D.items()})

crosscheck = {}
for key, e in summary.items():
    if e["oom"] or e["median_stage3_host_to_device_s"] is None:
        continue
    predicted = (e["param_bytes"] / 2**30) / H2D[e["device_index"]]
    observed = e["median_stage3_host_to_device_s"]
    crosscheck[key] = {
        "param_gib": e["param_bytes"] / 2**30,
        "nb03_h2d_gib_s": H2D[e["device_index"]],
        "predicted_pcie_s": predicted,
        "observed_stage3_s": observed,
        "observed_over_predicted": observed / predicted,
        "effective_gib_s": e["median_stage3_gib_s"],
    }
    print(f"{key:<52} predicted {predicted:6.3f}s  observed {observed:6.3f}s  "
          f"ratio {observed / predicted:5.2f}x  effective {e['median_stage3_gib_s']:5.2f} GiB/s")

notebook 03 H2D: {0: 6.36, 1: 2.81}
Qwen/Qwen3-Embedding-4B@cuda:0                       predicted  1.178s  observed  1.316s  ratio  1.12x  effective  5.69 GiB/s
intfloat/e5-small-v2@cuda:0                          predicted  0.010s  observed  0.020s  ratio  2.08x  effective  3.06 GiB/s
intfloat/e5-small-v2@cuda:1                          predicted  0.022s  observed  0.035s  ratio  1.59x  effective  1.77 GiB/s
sentence-transformers/all-MiniLM-L6-v2@cuda:0        predicted  0.007s  observed  0.013s  ratio  1.90x  effective  3.34 GiB/s
sentence-transformers/all-MiniLM-L6-v2@cuda:1        predicted  0.015s  observed  0.023s  ratio  1.56x  effective  1.80 GiB/s


## Two simultaneous cold loads vs two sequential ones

This is the measurement `max_concurrent_loads` needs and nothing else will
answer. The pairs are chosen to stress the thing in question:

- **cross-device**: the 4B on cuda:0 and a small model on cuda:1. Separate
  PCIe links, shared host CPU and page cache.
- **same-device**: two small models both on cuda:0, contending for one link
  and one allocator.

If concurrency degrades each load by less than it saves in makespan, a cap
above 1 is worth having. If a second load makes the first disproportionately
slower, the cap belongs at 1.

In [7]:
from concurrent.futures import ThreadPoolExecutor

PAIRS = [
    ("cross-device", [("Qwen/Qwen3-Embedding-4B", 0), ("intfloat/e5-small-v2", 1)]),
    ("same-device", [("intfloat/e5-small-v2", 0),
                     ("sentence-transformers/all-MiniLM-L6-v2", 0)]),
]

concurrency = {}
for label, pair in PAIRS:
    # Sequential baseline: each load alone, nothing else running.
    seq = []
    for model_id, device_index in pair:
        t0 = time.perf_counter()
        r = run_worker(model_id, SNAPSHOTS[model_id], device_index)
        seq.append({"model_id": model_id, "device_index": device_index,
                    "wall_s": time.perf_counter() - t0, "oom": r["oom"]})
    seq_makespan = sum(s["wall_s"] for s in seq)

    # Concurrent: both started together, in separate processes.
    def one(spec):
        model_id, device_index = spec
        t0 = time.perf_counter()
        r = run_worker(model_id, SNAPSHOTS[model_id], device_index)
        return {"model_id": model_id, "device_index": device_index,
                "wall_s": time.perf_counter() - t0, "oom": r["oom"]}

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=2) as pool:
        con = list(pool.map(one, pair))
    con_makespan = time.perf_counter() - t0

    per_load = [c["wall_s"] / s["wall_s"] for c, s in zip(con, seq)]
    concurrency[label] = {
        "pair": [{"model_id": m, "device_index": d} for m, d in pair],
        "sequential_each_s": [s["wall_s"] for s in seq],
        "sequential_makespan_s": seq_makespan,
        "concurrent_each_s": [c["wall_s"] for c in con],
        "concurrent_makespan_s": con_makespan,
        "per_load_slowdown": per_load,
        "makespan_speedup": seq_makespan / con_makespan,
    }
    print(f"\n{label}:")
    for s, c, ratio in zip(seq, con, per_load):
        print(f"  {s['model_id']:<45} cuda:{s['device_index']}  "
              f"alone {s['wall_s']:6.2f}s -> together {c['wall_s']:6.2f}s  ({ratio:.2f}x)")
    print(f"  makespan {seq_makespan:6.2f}s sequential -> {con_makespan:6.2f}s concurrent "
          f"({seq_makespan / con_makespan:.2f}x)")


cross-device:
  Qwen/Qwen3-Embedding-4B                       cuda:0  alone  14.19s -> together  13.50s  (0.95x)
  intfloat/e5-small-v2                          cuda:1  alone   4.92s -> together   5.16s  (1.05x)
  makespan  19.12s sequential ->  13.50s concurrent (1.42x)



same-device:
  intfloat/e5-small-v2                          cuda:0  alone   4.98s -> together   5.66s  (1.14x)
  sentence-transformers/all-MiniLM-L6-v2        cuda:0  alone   4.80s -> together   5.54s  (1.15x)
  makespan   9.78s sequential ->   5.66s concurrent (1.73x)


## Results

Filled in from the run above. Conclusions in the next cell are written
against these numbers, not around them.

In [8]:
# --- Evidence file. Anything that reaches README/DEPLOY comes from here. ---
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pyproject.toml").exists())
out_dir = root / "dev" / "output"
out_dir.mkdir(parents=True, exist_ok=True)

vram_end = {d["index"]: torch.cuda.mem_get_info(d["index"])[0] / 2**30 for d in DEVICES}

payload = {
    "purpose": "decompose cold model load latency; calibrate default_keep_alive_s "
               "and max_concurrent_loads",
    "driver": DRIVER,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "devices": DEVICES,
    "vram_free_end_gib": vram_end,
    "other_vram_users": OTHER_VRAM_USERS,
    "page_cache_eviction": {
        "method": "posix_fadvise(POSIX_FADV_DONTNEED) per checkpoint file",
        "reason_not_drop_caches": "drop_caches needs root; sudo requires a password "
                                  "on this host and was not used",
        "verified_per_run": True,
    },
    "measurement": {
        "process_isolation": "one fresh subprocess per measurement, so kernel "
                             "autotune caches and CUDA context are never shared",
        "runs": RUNS,
        "stages_summed": False,
    },
    "triton_python_headers": TRITON_HEADER_NOTE,
    "nb03_h2d_gib_s": H2D,
    "per_model_device": summary,
    "pcie_crosscheck": crosscheck,
    "concurrent_vs_sequential": concurrency,
    "raw_measurements": measurements,
}
path = out_dir / "model_load_results.json"
path.write_text(json.dumps(payload, indent=2, default=str))
print(f"wrote {path} ({path.stat().st_size} bytes)")

wrote /home/luxor/Programming/embedx/dev/output/model_load_results.json (39414 bytes)


## Conclusions

*(written after the run; see the cells above for the numbers)*